In [ ]:
# Cell 1 — install dependencies
!pip install -q streamlit pyngrok pillow matplotlib numpy pandas opencv-python-headless
# Optional UI helper (if you want it later). We avoid streamlit_option_menu to keep things simple.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 118.7 MB/s eta 0:00:00


In [ ]:
# Cell 2 — mount Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Cell 3 — configure paths and ngrok token
# EDIT this to match the exact path of your model on Google Drive.
# Example: "/content/drive/MyDrive/your_folder/best_model.h5"
MODEL_PATH = "/content/drive/MyDrive/Capstone Project CSE_4/front end design/best_model.h5"

# Your ngrok authtoken (you supplied one). Keep secret.
NGROK_AUTH_TOKEN = "33DqnHToxFHWe6zNg6YVctE1XEy_TvyGrAWbZw9mR8uvXWsQ"

# Safety checks (don't change)
import os
print("Model exists at:", os.path.exists(MODEL_PATH), "->", MODEL_PATH)


Model exists at: True -> /content/drive/MyDrive/Capstone Project CSE_4/front end design/best_model.h5


In [ ]:
# Cell 4 — write the streamlit app
app_code = r'''
import streamlit as st
import os
from PIL import Image
import numpy as np
import tensorflow as tf
import pandas as pd
import streamlit.components.v1 as components

# ------------- Settings -------------
MODEL_PATH = os.environ.get("MODEL_PATH", "")
CLASS_MAPPING = {
    0: 'Actinic keratoses (akiec)',
    1: 'Basal cell carcinoma (bcc)',
    2: 'Benign keratosis-like lesions (bkl)',
    3: 'Dermatofibroma (df)',
    4: 'Melanocytic nevi (nv)',
    5: 'Vascular lesions (vasc)',
    6: 'Melanoma (mel)'
}

# ------------- Load model safely -------------
@st.cache_resource(show_spinner=False)
def load_model_safe(path):
    if not path or not os.path.exists(path):
        raise FileNotFoundError(f"Model file not found at: {path}")
    model = tf.keras.models.load_model(path, compile=False)
    return model

# ------------- Helper: preprocess -------------
def preprocess_image_pil(img_pil, target_size):
    img = img_pil.convert("RGB")
    if target_size is not None:
        img = img.resize(target_size, resample=Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32)
    arr = arr / 255.0
    arr = np.expand_dims(arr, axis=0)
    return arr

# ------------- Guidance text -------------
def guidance_for(class_name):
    malignant = ['Melanoma (mel)', 'Basal cell carcinoma (bcc)']
    precancer = ['Actinic keratoses (akiec)']
    if any(m in class_name for m in malignant):
        return ("This prediction indicates a potentially malignant lesion. "
                "Seek urgent dermatology evaluation and consider biopsy.")
    if any(p in class_name for p in precancer):
        return ("This lesion can be precancerous. Please consult a dermatologist for assessment.")
    return ("This lesion appears likely to be benign. Continue to monitor it for changes. "
            "Consult a dermatologist if you notice rapid change, bleeding, or pain.")

# ------------- Streamlit layout -------------
st.set_page_config(page_title="DermDetect — Professional", layout="wide")

# Top navigation as buttons
col1, col2 = st.columns([1,3])
with col1:
    st.markdown("<h2 style='margin:0'>DermDetect</h2>", unsafe_allow_html=True)
with col2:
    nav = st.radio("", ["Welcome", "Diagnosis"], index=0, horizontal=True)

# ---------- Welcome page ----------
if nav == "Welcome":
    st.markdown("<h3>Welcome to DermDetect</h3>", unsafe_allow_html=True)
    st.markdown("A professional decision-support tool. This demo runs the model locally (no uploads to third-party servers).")
    st.markdown("---")

    # Scrolling flashcards implemented via small HTML widget
    facts = [
        "Early detection dramatically improves outcomes for melanoma.",
        "AI assists clinicians by prioritizing suspicious lesions for review.",
        "Diverse datasets are crucial to avoid diagnostic bias across skin types.",
        "This tool is for informational purposes and does not replace clinical judgment."
    ]
    # build minimal HTML carousel (auto-advancing)
    html_items = "".join(f"<div class='card'>{f}</div>" for f in facts)
    html = f"""
    <style>
      .carousel {{ width:100%; overflow:hidden; border-radius:8px; background:#fff; border:1px solid #e6e6e6; padding:20px; }}
      .cards {{ display:flex; gap:20px; width:100%; animation: slide {len(facts)*6}s linear infinite; }}
      .card {{ min-width:100%; padding:20px; box-sizing:border-box; font-size:20px; color:#222; }}
      @keyframes slide {{
        0% {{ transform: translateX(0%); }}
        20% {{ transform: translateX(0%); }}
        25% {{ transform: translateX(-100%); }}
        45% {{ transform: translateX(-100%); }}
        50% {{ transform: translateX(-200%); }}
        70% {{ transform: translateX(-200%); }}
        75% {{ transform: translateX(-300%); }}
        95% {{ transform: translateX(-300%); }}
        100% {{ transform: translateX(0%); }}
      }}
    </style>
    <div class="carousel">
      <div class="cards">
        {html_items}
      </div>
    </div>
    """
    components.html(html, height=180)

    st.markdown("### Facts & Background")
    st.markdown("- Dataset: HAM10000 / ISIC (combined in training pipeline).")
    st.markdown("- Model: Keras model. This interface loads a pre-trained model from your Drive.")
    st.markdown("---")

    st.markdown("### Visuals")
    st.image("https://images.unsplash.com/photo-1581091871776-7ca61d00b1d8?q=80&w=1600&auto=format&fit=crop", caption="Clinical dermatology", use_column_width=True)

# ---------- Diagnosis page ----------
else:
    st.markdown("<h3>Diagnosis</h3>", unsafe_allow_html=True)
    st.markdown("Upload a clear image of a single skin lesion. For best results use a dermatoscope or close well-lit photo.")
    st.write("Model path:", MODEL_PATH if MODEL_PATH else "(not configured)")

    # Attempt to load model
    try:
        model = load_model_safe(MODEL_PATH)
        # infer model input size
        try:
            input_shape = model.input_shape[1:3]
            if input_shape[0] is None:
                input_shape = (28, 28)
        except Exception:
            input_shape = (28,28)
    except Exception as e:
        st.error(f"Could not load model: {e}")
        st.stop()

    uploaded = st.file_uploader("Upload lesion image (jpg/png)", type=["jpg","jpeg","png"])

    if uploaded is not None:
        img = Image.open(uploaded)
        st.image(img, caption="Uploaded image", use_column_width=True)
        try:
            x = preprocess_image_pil(img, target_size=input_shape)
            preds = model.predict(x)
            pred_idx = int(np.argmax(preds, axis=1)[0])
            prob = float(np.max(preds) * 100)

            class_name = CLASS_MAPPING.get(pred_idx, f"Class {pred_idx}")
            st.subheader("Prediction")
            st.markdown(f"**Predicted class:** {class_name}")
            st.markdown(f"**Confidence:** {prob:.2f}%")
            st.markdown("**Preliminary guidance:**")
            st.info(guidance_for(class_name))

            st.markdown("---")
            st.markdown("**Model outputs (top 3)**")
            topk = np.argsort(preds[0])[-3:][::-1]
            for k in topk:
                st.write(f"{CLASS_MAPPING.get(int(k),'class')} — {float(preds[0][k])*100:.2f}%")

        except Exception as e:
            st.error(f"Prediction error: {e}")
'''
open('app.py','w', encoding='utf-8').write(app_code)
print("Wrote app.py")


Wrote app.py


In [ ]:
# Cell 5 — set MODEL_PATH environment variable for the app to use
import os
# Ensure MODEL_PATH from previous cell is available here (edit in that cell if needed)
os.environ["MODEL_PATH"] = MODEL_PATH
print("MODEL_PATH set to:", os.environ["MODEL_PATH"])


MODEL_PATH set to: /content/drive/MyDrive/Capstone Project CSE_4/front end design/best_model.h5


In [ ]:
# Cell 6 — start streamlit (runs in background)
# Note: this will run in the notebook environment. Logs will appear in the cell output.
get_ipython().system_raw("nohup streamlit run app.py --server.port 8501 --server.enableCORS false &> streamlit.log &")
print("Streamlit launched, logs -> streamlit.log")


Streamlit launched, logs -> streamlit.log


In [ ]:
# Cell 7 — configure ngrok and get public URL
from pyngrok import ngrok, conf
# set your authtoken (the token you provided)
NGROK_AUTH_TOKEN = "33DqnHToxFHWe6zNg6YVctE1XEy_TvyGrAWbZw9mR8uvXWsQ"
conf.get_default().auth_token = NGROK_AUTH_TOKEN
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# kill old tunnels (safety)
ngrok.kill()

# open new tunnel for port 8501
public_url = ngrok.connect(8501, "http").public_url
print("Streamlit public URL:", public_url)


Streamlit public URL: https://daniell-mesenteronic-sherlyn.ngrok-free.dev
